In [38]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
import seaborn as sns


ENDERECO_DADOS = 'https://www.ispdados.rj.gov.br/Arquivos/BaseDPEvolucaoMensalCisp.csv'


try:
    print('Obtendo os dados...')
    df_roubos = pd.read_csv(ENDERECO_DADOS, sep=';', encoding='iso-8859-1')
    print('Dados obtidos com sucesso!')
    print(df_roubos.head(10))
    print(df_roubos['regiao'].unique())




except Exception as e:
    print(f'Erro ao carregar o arquivo: {e} ')

Obtendo os dados...
Dados obtidos com sucesso!
   cisp  mes   ano  mes_ano  aisp  risp           munic    mcirc   regiao  \
0     1    1  2003  2003m01     5     1  Rio de Janeiro  3304557  Capital   
1     4    1  2003  2003m01     5     1  Rio de Janeiro  3304557  Capital   
2     5    1  2003  2003m01     5     1  Rio de Janeiro  3304557  Capital   
3     6    1  2003  2003m01     1     1  Rio de Janeiro  3304557  Capital   
4     7    1  2003  2003m01     1     1  Rio de Janeiro  3304557  Capital   
5     9    1  2003  2003m01     2     1  Rio de Janeiro  3304557  Capital   
6    10    1  2003  2003m01     2     1  Rio de Janeiro  3304557  Capital   
7    12    1  2003  2003m01    19     1  Rio de Janeiro  3304557  Capital   
8    13    1  2003  2003m01    19     1  Rio de Janeiro  3304557  Capital   
9    14    1  2003  2003m01    23     1  Rio de Janeiro  3304557  Capital   

   hom_doloso  ...  cmp  cmba  ameaca  pessoas_desaparecidas  \
0           0  ...  NaN   NaN      21    

In [39]:
# Tratamento dos dados
try:
    df_roubos = df_roubos[['cisp', 'regiao', 'munic', 'roubo_veiculo']]
    correcao = df_roubos['regiao'].str.startswith('Grande Niter', na=False)
    df_roubos.loc[correcao, 'regiao'] = 'Grande Niterói'
    print(df_roubos['regiao'].unique())


except Exception as e:
    print(f'Erro ao tratar os dados: {e}')

<StringArray>
['Capital', 'Baixada Fluminense', 'Interior', 'Grande Niterói']
Length: 4, dtype: str


In [40]:
# Processando as informações
try:
    df_roubos = df_roubos.groupby(['cisp', 'regiao', 'munic'], as_index=False)['roubo_veiculo'].sum()
    display(df_roubos)
except Exception as e:
    print(f'Erro ao processar os dados: {e}')

,cisp,regiao,munic,roubo_veiculo
0,1,Capital,Rio de Janeiro,573
1,4,Capital,Rio de Janeiro,2201
2,5,Capital,Rio de Janeiro,1365
3,6,Capital,Rio de Janeiro,4632
4,7,Capital,Rio de Janeiro,2016
...,...,...,...,...
141,159,Interior,Cachoeiras de Macacu,286
142,165,Interior,Mangaratiba,500
143,166,Interior,Angra dos Reis,970
144,167,Interior,Paraty,106


In [41]:
# Calculando as medidas: Média, Mediana, Total, Quartil
try:
    array_roubos = np.array(df_roubos['roubo_veiculo'])
    media = np.mean(array_roubos)
    mediana = np.median(array_roubos)
    total = np.sum(array_roubos)
    q1 = np.quantile(array_roubos, 0.25)
    q3 = np.quantile(array_roubos, 0.75)
    
    print('Media: ', media)
    print('Mediana: ', mediana)
    print('Total: ', total)
    print('Quartil 1: ', q1)
    print('Quartil 3: ', q3)
    
except Exception as e:
    print(f'Erro ao calcular a medida: {e}')

Media:  4907.698630136986
Mediana:  945.5
Total:  716524
Quartil 1:  93.25
Quartil 3:  7342.25


In [42]:
# Identificando os MAIS e os MENOS

try:
    df_maiores = df_roubos[df_roubos['roubo_veiculo'] > q3].copy()
    df_maiores['flags'] = 'mais'
    
    df_menores = df_roubos[df_roubos['roubo_veiculo'] < q1].copy()
    df_menores['flags'] = 'menos'


# Concatenando os dois dataframes
    
    df_roubos_flags = pd.concat([df_maiores, df_menores], ignore_index=True)
    display(df_roubos_flags)

except Exception as e:
    print(f'Erro ao identificar os MAIS e os MENOS: {e}')

,cisp,regiao,munic,roubo_veiculo,flags
0,17,Capital,Rio de Janeiro,7674,mais
1,20,Capital,Rio de Janeiro,9033,mais
2,21,Capital,Rio de Janeiro,19637,mais
3,22,Capital,Rio de Janeiro,15106,mais
4,23,Capital,Rio de Janeiro,10759,mais
...,...,...,...,...,...
69,155,Interior,São Sebastião do Alto,29,menos
70,156,Interior,Santa Maria Madalena,47,menos
71,157,Interior,Trajano de Morais,21,menos
72,158,Interior,Bom Jardim,21,menos


In [43]:
# Exportando os dados

try:
    df_roubos_flags.to_csv('df_roubos_flags.csv', index=False)
    
# Exportando para excel

    df_roubos_flags.to_excel('df_roubos_flags.xlsx', index=False)

except Exception as e:
    print(f'Erro ao exportar os dados: {e}')